In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

    
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
pip install transformers

In [ ]:
def path_expander(row):
    img_dir = '/kaggle/input/ristek-datathon-2022/media/Image/' + row
    return img_dir

def stratified_sample_df(df, col, n_samples):
    sample_train, drop_train = train_test_split(df, train_size=n_samples, stratify=df["label"], shuffle=True, random_state=42)
    return sample_train, drop_train

def process_image(df):
    images = []
    texts = []
    for index, row in df.iterrows():
        img = plt.imread(f'/kaggle/input/ristek-datathon-2022/media/Image/{row["media"]}')

        #       # Detectron expects BGR images
        #       img_bgr = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
        images.append(img)

        text = row["text"]
        texts.append(text)
    return images, texts

In [ ]:
train_df = pd.read_csv(f"/kaggle/input/ristek-datathon-2022/train.csv")
test_df = pd.read_csv(f"/kaggle/input/ristek-datathon-2022/test.csv")

train_df.head()

In [ ]:
train_df['img_dir'] = train_df['media'].apply(path_expander)
test_df['img_dir'] = test_df['media'].apply(path_expander)

In [ ]:
train_df.info()

In [ ]:
import pandas as pd
from torchvision import transforms
from PIL import Image
from transformers import BertTokenizer

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Image preprocessing transformations
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to 224x224 for ResNet
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # ImageNet normalization
])

# Function to preprocess text
def preprocess_text(text):
    # Tokenize text
    inputs = tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    input_ids = inputs['input_ids']
    attention_mask = inputs['attention_mask']
    return input_ids, attention_mask

# Function to preprocess image
def preprocess_image(img_dir):
    image = Image.open(img_dir)
    image = image_transform(image)
    return image

# Preprocess data
processed_data = []
for idx, row in train_df.iterrows():
    input_ids, attention_mask = preprocess_text(row['text'])
    image = preprocess_image(row['img_dir'])
    label = row['label']
    processed_data.append((input_ids, attention_mask, image, label))


In [ ]:
processed_data[0]

In [ ]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
from torchvision import models

class MultimodalFusion(nn.Module):
    def __init__(self, num_classes):
        super(MultimodalFusion, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.fc = nn.Linear(2048 + 768, num_classes)  # Fusion layer
        
    def forward(self, images, input_ids, attention_mask):
        # Extract visual features
        visual_features = self.resnet(images)
        
        # Extract textual features
        _, textual_features = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        
        # Perform fusion
        fused_features = torch.cat((visual_features, textual_features), dim=1)
        
        # Classification layer
        output = self.fc(fused_features)
        return output

# Example usage
num_classes = 2  # Number of classes in the classification task
model = MultimodalFusion(num_classes)

# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
import torch.nn as nn

# Define hyperparameters
batch_size = 8
num_epochs = 10
learning_rate = 0.001

# Convert processed data to tensors
input_ids_tensor = torch.stack([item[0] for item in processed_data])
attention_mask_tensor = torch.stack([item[1] for item in processed_data])
image_tensor = torch.stack([item[2] for item in processed_data])
labels_tensor = torch.tensor([item[3] for item in processed_data])

# Create a DataLoader
dataset = TensorDataset(input_ids_tensor, attention_mask_tensor, image_tensor, labels_tensor)
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Define the model, loss function, and optimizer
model = MultimodalFusion(num_classes=2)  # Adjust num_classes based on your task
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    
    for input_ids, attention_mask, images, labels in data_loader:
        # Forward pass
        outputs = model(images, input_ids, attention_mask)
        
        # Compute loss
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # Print average loss for the epoch
    print(f"Epoch {epoch+1}, Average Loss: {total_loss / len(data_loader)}")

    # Optionally, evaluate the model on a validation set here
